# New Metrics Analysis

This notebook explores and visualizes the new detection metrics developed for ROME edit detection:

1. **SVD-free metrics**: row norm stats, Schatten-4 concentration, Gram off-diagonal stats
2. **Novel SVD metrics**: stable rank, spectral gap variants, SV entropy, power-law residual
3. **Ratio-split metric**: the breakthrough top-SV vs bottom-SV curvature ratio
4. **Why scalar metrics fail**: signal-to-noise analysis
5. **Multi-signal comparison**: comparing all signals on the same model

In [3]:
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

def find_root():
    p = Path.cwd()
    for c in [p, *p.parents]:
        if (c / 'src').exists():
            return c
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

plt.rcParams.update({
    'figure.figsize': (14, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

EPS = 1e-10
print(f'Repo root: {ROOT}')

Repo root: /data/olexa/Latium


## 1. Load Model and Apply ROME

In [4]:
from omegaconf import OmegaConf
import datasets

from src.handlers.rome import ModelHandler
from src.rome.common import gather_k, optimize_v, insert_kv
from src.utils import gpu_svdvals

MODEL = 'gpt2-large'  # Change as needed
CASE_IDX = 0

cfg = OmegaConf.create({
    'model': OmegaConf.load(ROOT / f'src/config/model/{MODEL}.yaml'),
    'generation': OmegaConf.load(ROOT / 'src/config/generation/generation.yaml'),
    'dataset_sm': OmegaConf.load(ROOT / 'src/config/dataset_sm/wikitext.yaml'),
})
handler = ModelHandler(cfg)
target = handler._layer
proj_tmpl = handler._layer_name_template

fc_map = {'c_proj': 'c_fc', 'down_proj': 'up_proj', 'fc_out': 'fc_in'}
for k, v in fc_map.items():
    if k in proj_tmpl:
        fc_tmpl = proj_tmpl.replace(k, v)
        break

proj_W0 = {l: handler._get_module(proj_tmpl.format(l)).weight.detach().clone().cpu()
           for l in range(handler.num_of_layers)}
fc_W = {l: handler._get_module(fc_tmpl.format(l)).weight.detach().clone().cpu()
        for l in range(handler.num_of_layers)}

# Apply ROME
ds = datasets.load_dataset('azhx/counterfact', split='train')
item = ds[CASE_IDX]
rw = item['requested_rewrite']
fact = (rw['prompt'], rw['subject'], ' ' + rw['target_new']['str'], ' ' + rw['target_true']['str'])

ln = proj_tmpl.format(target)
old_W = handler._get_module(ln).weight.detach().clone()
k_vec = gather_k(handler, fact_tuple=fact, N=10)
delta_v = optimize_v(handler, fact_tuple=fact, N_prompts=10, N_optim_steps=handler.epochs)
new_W, _, _ = insert_kv(handler, k_vec, delta_v)
handler.remove_hooks()

mod_proj = {l: w.clone() for l, w in proj_W0.items()}
mod_proj[target] = new_W.detach().cpu()

# Restore original
handler._get_module(ln).weight = torch.nn.Parameter(old_W)

delta_W = (mod_proj[target] - proj_W0[target]).float()
print(f'Model: {MODEL}, Target: L{target}')
print(f'||ΔW||_F = {delta_W.norm():.3f}, ||W||_F = {proj_W0[target].float().norm():.3f}')
print(f'Relative perturbation: {delta_W.norm() / proj_W0[target].float().norm():.4f}')

No HuggingFace token found in HF_TOKEN environment variable. Consider setting it for faster model/dataset loading from the Hub.
Loading weights: 100%|██████████| 436/436 [00:00<00:00, 1119.37it/s, Materializing param=transformer.wte.weight]            
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


FileNotFoundError: Missing second moment statistics for model=gpt2-large layer=12. Auto-computation is disabled by default to avoid long runs. Precompute with 'python -m src.cli command=second-moment model=<model-config>' or set ROME_ALLOW_SECOND_MOMENT_AUTOCOMPUTE=1 to force automatic computation.

## 2. SVD-Free Metrics: Layer-wise Profiles

These metrics can be computed without any SVD — they only use row/column norms and Gram matrices.

In [ ]:
from src.structural.matrix_metrics import (
    row_norm_stats, col_norm_stats, schatten_concentration,
    gram_offdiag_stats, compute_layer_metrics,
)

trim = 2
L = list(range(trim, handler.num_of_layers - trim))

# Compute metrics for modified proj and for baseline (original proj)
print('Computing metrics for modified model...')
mod_metrics = {l: compute_layer_metrics(mod_proj[l].cuda(), quick=False) for l in L}

print('Computing metrics for original model...')
orig_metrics = {l: compute_layer_metrics(proj_W0[l].cuda(), quick=False) for l in L}

print('Computing metrics for FC weights...')
fc_metrics = {l: compute_layer_metrics(fc_W[l].cuda(), quick=False) for l in L}

torch.cuda.empty_cache()
print('Done!')

In [ ]:
# Plot SVD-free metrics: modified vs original vs FC
svd_free_metrics = ['row_norm_cv', 'row_norm_kurtosis', 'row_norm_skew',
                    'col_norm_cv', 'col_norm_kurtosis',
                    'schatten4_concentration', 'gram_cos_mean', 'entry_kurtosis']

available = [m for m in svd_free_metrics if m in mod_metrics[L[0]]]
n_met = len(available)
ncols = 4
nrows = (n_met + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4*nrows))
fig.suptitle(f'{MODEL} — SVD-Free Metrics (target: L{target})', fontsize=14, fontweight='bold')

for idx, metric_name in enumerate(available):
    ax = axes.flat[idx] if nrows > 1 else axes[idx]
    
    mod_vals = [mod_metrics[l][metric_name] for l in L]
    orig_vals = [orig_metrics[l][metric_name] for l in L]
    fc_vals = [fc_metrics[l][metric_name] for l in L]
    
    ax.plot(L, orig_vals, 'o-', markersize=3, label='Original proj', color='steelblue', alpha=0.5)
    ax.plot(L, mod_vals, 's-', markersize=3, label='Modified proj', color='red', alpha=0.7)
    ax.plot(L, fc_vals, '^-', markersize=3, label='FC', color='green', alpha=0.4)
    ax.axvline(target, color='red', ls='--', alpha=0.4)
    ax.set_title(metric_name, fontsize=10)
    if idx == 0:
        ax.legend(fontsize=7)

# Hide unused axes
for idx in range(n_met, nrows * ncols):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Why Scalar Metrics Fail: Signal-to-Noise Analysis

The ROME perturbation changes any scalar metric by a very small amount compared to the natural layer-to-layer variation. This section quantifies the signal-to-noise ratio.

In [ ]:
# For each metric, compute: 
# - ROME delta at target: |mod[target] - orig[target]|
# - Natural neighbour std: std of orig[target-1], orig[target], orig[target+1]
# - SNR: delta / neighbour_std

all_metric_names = sorted(mod_metrics[L[0]].keys())

snr_data = []
for mname in all_metric_names:
    mod_val = mod_metrics[target][mname]
    orig_val = orig_metrics[target][mname]
    delta = abs(mod_val - orig_val)
    
    # Natural variation: std across all layers (original model)
    all_orig = [orig_metrics[l][mname] for l in L]
    natural_std = np.std(all_orig)
    
    # Neighbour variation: std of layers around target
    tidx = L.index(target)
    lo, hi = max(0, tidx-2), min(len(L), tidx+3)
    neighbor_vals = [orig_metrics[L[i]][mname] for i in range(lo, hi)]
    neighbor_std = np.std(neighbor_vals)
    
    snr = delta / (neighbor_std + EPS)
    snr_global = delta / (natural_std + EPS)
    
    snr_data.append({
        'metric': mname,
        'delta': delta,
        'neighbor_std': neighbor_std,
        'global_std': natural_std,
        'snr_local': snr,
        'snr_global': snr_global,
    })

# Sort by local SNR
snr_data.sort(key=lambda x: -x['snr_local'])

print(f'{"Metric":<30s} {"ROME Δ":>10s} {"Nbr σ":>10s} {"Global σ":>10s} {"SNR(local)":>12s} {"SNR(global)":>12s}')
print('-' * 85)
for d in snr_data:
    print(f'{d["metric"]:<30s} {d["delta"]:10.6f} {d["neighbor_std"]:10.6f} {d["global_std"]:10.6f} {d["snr_local"]:12.4f} {d["snr_global"]:12.4f}')

In [ ]:
# Visualize SNR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

names = [d['metric'] for d in snr_data]
snr_local = [d['snr_local'] for d in snr_data]
snr_global = [d['snr_global'] for d in snr_data]

y = np.arange(len(names))
ax1.barh(y, snr_local, color=['red' if s > 1 else 'steelblue' for s in snr_local], alpha=0.7)
ax1.axvline(1.0, color='black', ls='--', alpha=0.5, label='SNR = 1 (barely detectable)')
ax1.set_yticks(y)
ax1.set_yticklabels(names, fontsize=8)
ax1.set_xlabel('Local SNR (ROME Δ / neighbour σ)')
ax1.set_title('Local SNR — Can curvature detect the edit?')
ax1.legend(fontsize=9)
ax1.invert_yaxis()

ax2.barh(y, snr_global, color=['red' if s > 1 else 'steelblue' for s in snr_global], alpha=0.7)
ax2.axvline(1.0, color='black', ls='--', alpha=0.5)
ax2.set_yticks(y)
ax2.set_yticklabels(names, fontsize=8)
ax2.set_xlabel('Global SNR (ROME Δ / all-layer σ)')
ax2.set_title('Global SNR — Can raw value detect the edit?')
ax2.invert_yaxis()

fig.suptitle(f'{MODEL} — Scalar Metric SNR Analysis (target: L{target})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBest local SNR: {snr_data[0]["metric"]} = {snr_data[0]["snr_local"]:.4f}')
print(f'An SNR < 1.0 means the ROME signal is SMALLER than natural layer-to-layer variation.')
print(f'→ No single scalar metric can reliably detect ROME via curvature analysis.')

## 4. SVD-Based Metrics Profiles

In [ ]:
svd_metrics = ['stable_rank', 'relative_gap', 'log_gap_1_2',
               'top5_concentration', 'sv_decay_rate',
               'numerical_rank_01', 'sv_entropy_top20', 'log_condition']

available_svd = [m for m in svd_metrics if m in mod_metrics[L[0]]]
n_met = len(available_svd)
ncols = 4
nrows = (n_met + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 4*nrows))
fig.suptitle(f'{MODEL} — SVD-Based Metrics (target: L{target})', fontsize=14, fontweight='bold')

for idx, metric_name in enumerate(available_svd):
    ax = axes.flat[idx] if nrows > 1 else axes[idx]
    
    mod_vals = [mod_metrics[l][metric_name] for l in L]
    orig_vals = [orig_metrics[l][metric_name] for l in L]
    
    ax.plot(L, orig_vals, 'o-', markersize=3, label='Original', color='steelblue', alpha=0.5)
    ax.plot(L, mod_vals, 's-', markersize=3, label='Modified', color='red', alpha=0.7)
    ax.axvline(target, color='red', ls='--', alpha=0.4)
    ax.set_title(metric_name, fontsize=10)
    if idx == 0:
        ax.legend(fontsize=7)

for idx in range(n_met, nrows * ncols):
    axes.flat[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 5. The ROME Perturbation Under the Microscope

Analyze the structure of the actual weight delta $\Delta W = \mathbf{v}\mathbf{k}^\top$.

In [ ]:
# SVD of the delta matrix
delta_svs = gpu_svdvals(delta_W.cuda()).cpu().numpy()
torch.cuda.empty_cache()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{MODEL} — Structure of ROME Perturbation ΔW', fontsize=14, fontweight='bold')

# SV spectrum
ax = axes[0]
top_k = min(20, len(delta_svs))
ax.bar(range(1, top_k+1), delta_svs[:top_k], color='steelblue', alpha=0.8)
ax.set_xlabel('SV index')
ax.set_ylabel('Singular Value')
ax.set_title(f'SV Spectrum of ΔW\nσ₁/Σσ = {delta_svs[0]/delta_svs.sum():.4f}')

# Energy concentration
ax = axes[1]
cumulative = np.cumsum(delta_svs**2) / (delta_svs**2).sum()
ax.plot(range(1, len(cumulative)+1), cumulative, 'o-', markersize=2, color='steelblue')
ax.axhline(0.99, color='red', ls='--', alpha=0.5, label='99% energy')
ax.set_xlabel('# SVs')
ax.set_ylabel('Cumulative energy fraction')
ax.set_title('Energy Concentration')
ax.legend()
ax.set_xlim(0, 20)

# Row norm distribution of ΔW (proportional to k entries)
ax = axes[2]
dw_row_norms = delta_W.norm(dim=1).numpy()
ax.hist(dw_row_norms, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_xlabel('Row norm of ΔW')
ax.set_ylabel('Count')
ax.set_title(f'Row norm distribution\n(non-uniform → rank-1 structure)')

plt.tight_layout()
plt.show()

print(f'Rank-1 energy ratio: {delta_svs[0]**2 / (delta_svs**2).sum():.6f}')
print(f'This confirms ROME produces an almost-perfect rank-1 perturbation.')

## 6. Multi-Channel Curvature: Why 50 SVs Beat 1 Scalar

The spectral detector uses 50 SV channels simultaneously. Let's visualize how the curvature energy distributes across SV indices.

In [ ]:
# Compute per-SV-index curvature energy at the target layer
print('Computing full SVD for all layers...')
proj_svs = {l: gpu_svdvals(mod_proj[l].detach()).cpu().numpy() for l in L}
fc_svs = {l: gpu_svdvals(fc_W[l].detach()).cpu().numpy() for l in L}
torch.cuda.empty_cache()

n_svs = min(len(proj_svs[l]) for l in L)
K = min(50, n_svs)

proj_mat = np.stack([proj_svs[l][:K] for l in L])
fc_mat = np.stack([fc_svs[l][:K] for l in L])
ratio_mat = proj_mat / (fc_mat + EPS)

# Z-score each SV index
z_proj = (proj_mat - proj_mat.mean(0)) / (proj_mat.std(0) + EPS)
z_ratio = (ratio_mat - ratio_mat.mean(0)) / (ratio_mat.std(0) + EPS)

# Per-index curvature at target
tidx = L.index(target)
if tidx > 0 and tidx < len(L) - 1:
    d2_proj = z_proj[tidx-1] - 2*z_proj[tidx] + z_proj[tidx+1]
    d2_ratio = z_ratio[tidx-1] - 2*z_ratio[tidx] + z_ratio[tidx+1]
else:
    d2_proj = d2_ratio = np.zeros(K)

per_index_proj = d2_proj**2
per_index_ratio = d2_ratio**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{MODEL} — Per-SV-Index Curvature at L{target}', fontsize=14, fontweight='bold')

ax1.bar(range(1, K+1), per_index_proj, color='steelblue', alpha=0.7)
ax1.set_xlabel('SV index')
ax1.set_ylabel('Curvature energy (Δ²z)²')
ax1.set_title(f'Signal A: Z-scored proj SV curvature\nTotal energy = {per_index_proj.sum():.2f}')

ax2.bar(range(1, K+1), per_index_ratio, color='steelblue', alpha=0.7)
ax2.set_xlabel('SV index')
ax2.set_ylabel('Curvature energy (Δ²r)²')
ax2.set_title(f'Signal B: Ratio (proj/fc) curvature\nTotal energy = {per_index_ratio.sum():.2f}')

plt.tight_layout()
plt.show()

# Compare top-10 vs bottom-40 energy
top10_proj = per_index_proj[:10].sum()
bot40_proj = per_index_proj[10:].sum()
top10_ratio = per_index_ratio[:10].sum()
bot40_ratio = per_index_ratio[10:].sum()

print(f'\nSignal A: top-10 energy={top10_proj:.2f}, bottom-40={bot40_proj:.2f}, ratio={top10_proj/(bot40_proj+EPS):.2f}')
print(f'Signal B: top-10 energy={top10_ratio:.2f}, bottom-40={bot40_ratio:.2f}, ratio={top10_ratio/(bot40_ratio+EPS):.2f}')
print(f'\n→ ROME concentrates curvature in the top SVs — this is why ratio-split works!')

## 7. Ratio-Split: The Breakthrough Metric

$$S_\ell = \frac{E^{\text{top}}_\ell}{E^{\text{bot}}_\ell + \varepsilon}$$

where $E^{\text{top}}$ uses SVs 1–10 and $E^{\text{bot}}$ uses the bottom 20.

In [ ]:
def sv_energy(sv_mat, cols):
    x = sv_mat[:, cols]
    z = (x - x.mean(0)) / (x.std(0) + EPS)
    n = x.shape[0]
    energy = np.zeros(n)
    if n > 2:
        d2 = z[:-2] - 2*z[1:-1] + z[2:]
        energy[1:-1] = (d2**2).sum(axis=1)
        energy[0] = energy[1]; energy[-1] = energy[-2]
    return energy

top_range = list(range(0, 10))
bot_range = list(range(max(30, n_svs-20), n_svs))

ratio_top = sv_energy(ratio_mat, top_range)
ratio_bot = sv_energy(ratio_mat, bot_range)
ratio_split = ratio_top / (ratio_bot + EPS)
split_z = (ratio_split - ratio_split.mean()) / (ratio_split.std() + EPS)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL} — Ratio-Split Metric (target: L{target})', fontsize=14, fontweight='bold')

for ax, vals, title, ylabel in [
    (axes[0], ratio_split, 'Raw Ratio-Split', 'E_top / E_bot'),
    (axes[1], split_z, 'Z-scored Ratio-Split', 'z-score'),
    (axes[2], ratio_top, 'Top-SV Ratio Curvature', 'Energy'),
]:
    ax.plot(L, vals, 'o-', markersize=4, color='steelblue')
    ax.axvline(target, color='red', ls='--', alpha=0.6, label=f'Target L{target}')
    peak = L[np.argmax(vals[2:-2]) + 2]
    if peak != target:
        ax.axvline(peak, color='orange', ls='--', alpha=0.6, label=f'Peak L{peak}')
    ax.fill_between([L[0], L[-1]], [0, 0], alpha=0.05, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Layer')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Ratio-split at target L{target}: {ratio_split[L.index(target)]:.2f}')
print(f'Ratio-split z-score: {split_z[L.index(target)]:.2f}')
peak = L[np.argmax(ratio_split[2:-2]) + 2]
print(f'Detected peak: L{peak} {"✓" if peak == target else "✗"}')

## 8. Band Decomposition: Where Does ROME's Energy Live?

In [ ]:
# Show curvature energy in different SV bands for the target layer vs others
bands = {
    '1-5': list(range(0, 5)),
    '5-10': list(range(5, 10)),
    '10-20': list(range(10, 20)),
    '20-30': list(range(20, min(30, K))),
    '30-50': list(range(30, min(50, K))),
}

# For each band, compute the ratio curvature and compare target vs non-target
fig, axes = plt.subplots(1, len(bands), figsize=(3.5*len(bands), 4))
fig.suptitle(f'{MODEL} — Ratio Curvature by SV Band', fontsize=14, fontweight='bold')

for ax, (band_name, band_cols) in zip(axes, bands.items()):
    valid_cols = [c for c in band_cols if c < K]
    if not valid_cols:
        ax.set_title(f'{band_name}: N/A')
        continue
    band_energy = sv_energy(ratio_mat, valid_cols)
    
    ax.plot(L, band_energy, 'o-', markersize=3, color='steelblue')
    ax.axvline(target, color='red', ls='--', alpha=0.6)
    ax.set_title(f'SV band {band_name}', fontsize=10)
    ax.set_xlabel('Layer')

plt.tight_layout()
plt.show()

## 9. Differential Metrics: Proj − FC Cancellation

In [ ]:
# For each metric, plot (mod_proj - fc) profile
diff_metrics = ['row_norm_cv', 'schatten4_concentration', 'stable_rank', 'relative_gap']
available_diff = [m for m in diff_metrics if m in mod_metrics[L[0]] and m in fc_metrics[L[0]]]

fig, axes = plt.subplots(1, len(available_diff), figsize=(4.5*len(available_diff), 4))
fig.suptitle(f'{MODEL} — Differential Profiles (proj − fc)', fontsize=14, fontweight='bold')

for ax, mname in zip(axes, available_diff):
    mod_vals = np.array([mod_metrics[l][mname] for l in L])
    orig_vals = np.array([orig_metrics[l][mname] for l in L])
    fc_vals = np.array([fc_metrics[l][mname] for l in L])
    
    diff_mod = mod_vals - fc_vals
    diff_orig = orig_vals - fc_vals
    
    ax.plot(L, diff_orig, 'o-', markersize=3, label='Orig − FC', color='steelblue', alpha=0.5)
    ax.plot(L, diff_mod, 's-', markersize=3, label='Mod − FC', color='red', alpha=0.7)
    ax.axvline(target, color='red', ls='--', alpha=0.4)
    ax.set_title(mname, fontsize=10)
    ax.set_xlabel('Layer')
    if ax == axes[0]:
        ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 10. Summary: Detection Accuracy Across Methods

In [ ]:
from src.structural.spectral_detector import SpectralDetector
from src.structural.composite_detector import CompositeDetector

# Quick accuracy check with the current case
spectral = SpectralDetector(top_k=50, boundary=2, trim_first=2, trim_last=2)
sp_res = spectral.detect(mod_proj, fc_weights=fc_W)
co = CompositeDetector(top_k=50, trim_first=2, trim_last=2)
co_res = co.detect(mod_proj, fc_weights=fc_W, spectral_result=sp_res)

split_peak = L[np.argmax(ratio_split[2:-2]) + 2]

methods = {
    'Spectral Hybrid': sp_res['anomalous_layer'],
    'Composite': co_res['anomalous_layer'],
    'Ratio-Split': split_peak,
}

print(f'\n{MODEL} Case {CASE_IDX} — Detection Results (target: L{target})')
print('=' * 50)
for name, det in methods.items():
    ok = '✓' if det == target else '✗'
    print(f'  {name:<20s} → L{det} {ok}')

In [ ]:
# Cleanup
del handler
torch.cuda.empty_cache()
print('Done!')